# Semana 2 — Dataset y DataLoaders
### Detector de Rostros de Futbolistas Argentinos · FIFA 2022

**Flujo del notebook:**
1. Obtener el dataset de rostros (desde Kaggle la primera vez, desde Drive las siguientes)
2. Extraer rostros con InsightFace *(solo primera vez — paralelizado con GPU automático)*
3. Filtrar jugadores argentinos y construir el DataFrame
4. Split estratificado 60 / 20 / 20 y guardar CSVs
5. Definir `FaceDataset` con **banderas individuales** de augmentation
6. Crear `DataLoader` de train, val y test con `build_dataloader`

In [ ]:
!pip install -q insightface onnxruntime-gpu gdown pandas scikit-learn tqdm

In [ ]:
import os
import json
import random
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
# ================================================================
# CONFIGURACION GLOBAL — Ajustar aqui antes de ejecutar
# ================================================================

# --- Control de flujo ---
SKIP_EXTRACTION  = True      # False solo en la primera corrida
DRIVE_ZIP_ID     = "1bf4dkkJWgH02et9NfVVxJis1X4C9nLsu"  # ID del ZIP en Drive

# --- Extraccion de rostros ---
USE_PADDING      = True      # Agregar margen alrededor del rostro
PADDING_PX       = 15        # Tamano del margen en pixeles
IMG_SIZE         = 224       # Tamano de salida (224x224)
MAX_WORKERS      = 4         # Workers paralelos para extraccion

# --- Augmentaciones de entrenamiento ---
USE_FLIP         = True      # RandomHorizontalFlip(p=0.5)
USE_COLOR_JITTER = True      # ColorJitter: brillo, contraste, saturacion

# --- DataLoader ---
BATCH_SIZE       = 32
NUM_WORKERS      = 2

# --- Rutas ---
DATASET_DIR  = "/content/FIFA_2022_ALL_PLAYERS"
FACES_DIR    = "/content/FIFA_2022_ONLY_FACES"
DATA_DIR     = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Configuracion activa:")
print(f"  SKIP_EXTRACTION  = {SKIP_EXTRACTION}")
print(f"  USE_PADDING      = {USE_PADDING}  (px={PADDING_PX})")
print(f"  USE_FLIP         = {USE_FLIP}")
print(f"  USE_COLOR_JITTER = {USE_COLOR_JITTER}")
print(f"  BATCH_SIZE       = {BATCH_SIZE}")

## Paso 1: Obtener el dataset de rostros

| `SKIP_EXTRACTION` | Accion |
|---|---|
| `False` *(primera vez)* | Descarga el dataset original de Kaggle (~360 MB) |
| `True` *(corridas siguientes)* | Descarga el ZIP de rostros ya procesados desde Drive |

> **Primera vez:** agregar el token de Kaggle en **Secretos de Colab** con el nombre `KAGGLE_TOKEN`.

In [ ]:
if not SKIP_EXTRACTION:
    from google.colab import userdata
    token = userdata.get("KAGGLE_TOKEN")
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/access_token", "w") as f:
        f.write(token)
    os.chmod("/root/.kaggle/access_token", 0o600)

    !kaggle datasets download -d soumendraprasad/fifa-2022-all-players-image-dataset
    !mkdir -p {DATASET_DIR}
    !unzip -q fifa-2022-all-players-image-dataset.zip -d {DATASET_DIR}
    print(f"Dataset original descargado en {DATASET_DIR}")

else:
    print("Descargando ZIP de rostros procesados desde Drive...")
    !gdown {DRIVE_ZIP_ID} -O /content/FIFA_2022_ONLY_FACES.zip
    !unzip -q /content/FIFA_2022_ONLY_FACES.zip -d /
    print(f"Rostros disponibles en {FACES_DIR}")

## Paso 2: Extraccion de rostros con InsightFace *(solo primera vez)*

- Detecta el rostro **mas grande** en cada imagen.
- Aplica el margen configurado con `USE_PADDING` / `PADDING_PX`.
- Usa **GPU automaticamente** si esta disponible; cae en CPU si no.
- Procesamiento **paralelo** con `ThreadPoolExecutor` — barra de progreso con `tqdm`.
- Al terminar, comprime el resultado y lo guarda en Google Drive.

In [ ]:
if not SKIP_EXTRACTION:
    import onnxruntime as ort
    from insightface.app import FaceAnalysis

    # Deteccion automatica GPU / CPU
    available = ort.get_available_providers()
    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if "CUDAExecutionProvider" in available
        else ["CPUExecutionProvider"]
    )
    print(f"Provider: {providers[0]}")

    app = FaceAnalysis(providers=providers)
    app.prepare(ctx_id=0, det_size=(640, 640))

    def extract_face(img, use_padding=USE_PADDING, padding_px=PADDING_PX):
        """Recorta el rostro mas grande de la imagen."""
        faces = app.get(img)
        if not faces:
            return None
        face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
        x1, y1, x2, y2 = map(int, face.bbox)
        h, w = img.shape[:2]
        if use_padding:
            x1 = max(0, x1 - padding_px)
            y1 = max(0, y1 - padding_px)
            x2 = min(w, x2 + padding_px)
            y2 = min(h, y2 + padding_px)
        crop = img[y1:y2, x1:x2]
        return cv2.resize(crop, (IMG_SIZE, IMG_SIZE)) if crop.size > 0 else None

    dataset_root = Path(DATASET_DIR)
    output_root  = Path(FACES_DIR)
    output_root.mkdir(parents=True, exist_ok=True)
    extensions   = {".jpg", ".jpeg", ".png"}
    img_files    = [p for p in dataset_root.rglob("*") if p.suffix.lower() in extensions]

    saved_count = 0

    def process_file(img_file):
        img = cv2.imread(str(img_file))
        if img is None:
            return False
        face = extract_face(img)
        if face is None:
            return False
        save_path = output_root / img_file.relative_to(dataset_root)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_path), face)
        return True

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_file, f): f for f in img_files}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Extrayendo rostros"):
            if fut.result():
                saved_count += 1

    print(f"Procesadas: {len(img_files)}  |  Rostros guardados: {saved_count}")

In [ ]:
if not SKIP_EXTRACTION:
    from google.colab import drive
    drive.mount("/content/drive")

    # Crear ZIP con rutas absolutas (mismo formato para extraccion posterior)
    !zip -r /content/FIFA_2022_ONLY_FACES.zip {FACES_DIR}

    import shutil
    dest = "/content/drive/MyDrive/FIFA_2022_ONLY_FACES.zip"
    shutil.copy("/content/FIFA_2022_ONLY_FACES.zip", dest)
    print(f"ZIP guardado en: {dest}")
    print("Obtener el ID del archivo en Drive y actualizar DRIVE_ZIP_ID en la configuracion.")

## Paso 3: DataFrame de Argentina

Se filtran solo los jugadores de la seleccion argentina y se construye el mapeo `label -> indice` necesario para el entrenamiento.

In [ ]:
def fix_encoding(name):
    try:
        return name.encode("latin1").decode("utf-8")
    except Exception:
        return name

faces_root = Path(FACES_DIR)
extensions = {".jpg", ".jpeg", ".png"}

records = []
for img_path in faces_root.rglob("*"):
    if img_path.suffix.lower() not in extensions:
        continue
    parts = img_path.relative_to(faces_root).parts
    if len(parts) < 2:
        continue
    if "argentina" not in parts[0].lower():
        continue
    records.append({
        "image_path": str(img_path.relative_to(faces_root)),
        "label":      fix_encoding(parts[1]),
    })

df = pd.DataFrame(records)

# Eliminar jugadores con menos de 5 imagenes
counts = df["label"].value_counts()
df = df[df["label"].isin(counts[counts >= 5].index)].reset_index(drop=True)

# Mapeo label <-> indice
all_labels   = sorted(df["label"].unique())
label_to_idx = {lbl: i for i, lbl in enumerate(all_labels)}
idx_to_label = {i: lbl for lbl, i in label_to_idx.items()}
df["label_idx"] = df["label"].map(label_to_idx)

print(f"Jugadores: {df['label'].nunique()}  |  Imagenes: {len(df)}")
print(df["label"].value_counts().to_string())

## Paso 4: Split 60 / 20 / 20 y guardado

Split estratificado por jugador. Se guardan en `DATA_DIR`:
- `train.csv`, `val.csv`, `test.csv`
- `label_to_idx.json` — necesario en produccion para decodificar la prediccion del modelo

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.40, stratify=df["label_idx"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_idx"], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df.to_csv(f"{DATA_DIR}/train.csv", index=False)
val_df.to_csv(  f"{DATA_DIR}/val.csv",   index=False)
test_df.to_csv( f"{DATA_DIR}/test.csv",  index=False)

with open(f"{DATA_DIR}/label_to_idx.json", "w", encoding="utf-8") as f:
    json.dump(label_to_idx, f, ensure_ascii=False, indent=2)

total = len(train_df) + len(val_df) + len(test_df)
print(f"TRAIN : {len(train_df):4d} imagenes  ({len(train_df)/total*100:.1f}%)")
print(f"VAL   : {len(val_df):4d} imagenes  ({len(val_df)/total*100:.1f}%)")
print(f"TEST  : {len(test_df):4d} imagenes  ({len(test_df)/total*100:.1f}%)")
print(f"TOTAL : {total}")
print(f"Archivos guardados en {DATA_DIR}/")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
splits = [(train_df, "Train"), (val_df, "Val"), (test_df, "Test")]

for ax, (split_df, name) in zip(axes, splits):
    cnt = split_df["label"].value_counts().sort_values()
    ax.barh(cnt.index, cnt.values, color="steelblue")
    ax.set_title(f"{name}  ({len(split_df)} imagenes)")
    ax.set_xlabel("Imagenes")
    ax.tick_params(axis="y", labelsize=7)

plt.suptitle("Distribucion de clases por subset", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Muestra un ejemplo por jugador (primeros 15)
sample_per_player = (
    df.groupby("label", group_keys=False)
      .apply(lambda g: g.sample(1, random_state=42))
      .reset_index(drop=True)
    .head(15)
)

fig, axes = plt.subplots(3, 5, figsize=(14, 9))
for ax, (_, row) in zip(axes.flatten(), sample_per_player.iterrows()):
    img = cv2.imread(str(faces_root / row["image_path"]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(row["label"], fontsize=7)
    ax.axis("off")

plt.suptitle("Muestra de rostros — Argentina", fontsize=13)
plt.tight_layout()
plt.show()

## Paso 5: Dataset y DataLoaders

`FaceDataset` recibe **una bandera por augmentation** para activarlas independientemente.
`build_dataloader` es el unico punto de entrada para construir cualquier loader.

| Bandera | Transformacion aplicada |
|---|---|
| `use_flip=True` | `RandomHorizontalFlip(p=0.5)` |
| `use_color_jitter=True` | `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)` |

Sin banderas: solo `Resize → ToTensor → Normalize` (usado en val y test).

In [ ]:
class FaceDataset(Dataset):

    def __init__(self, dataframe, root_dir, use_flip=False, use_color_jitter=False):
        self.df       = dataframe.reset_index(drop=True)
        self.root_dir = Path(root_dir)

        steps = [transforms.Resize((224, 224))]

        if use_flip:
            steps.append(transforms.RandomHorizontalFlip(p=0.5))
        if use_color_jitter:
            steps.append(transforms.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.2
            ))

        steps += [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ]
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root_dir / row["image_path"]).convert("RGB")
        return self.transform(img), int(row["label_idx"])

In [ ]:
def build_dataloader(df, root_dir,
                     use_flip=False, use_color_jitter=False,
                     batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    dataset = FaceDataset(df, root_dir,
                          use_flip=use_flip,
                          use_color_jitter=use_color_jitter)
    # shuffle activo solo cuando hay augmentation (train)
    shuffle = use_flip or use_color_jitter
    return DataLoader(dataset,
                      batch_size=batch_size,
                      shuffle=shuffle,
                      num_workers=num_workers,
                      pin_memory=True)

In [ ]:
train_loader = build_dataloader(
    train_df, FACES_DIR,
    use_flip=USE_FLIP,
    use_color_jitter=USE_COLOR_JITTER,
)
val_loader  = build_dataloader(val_df,  FACES_DIR)
test_loader = build_dataloader(test_df, FACES_DIR)

print(f"train_loader : {len(train_loader.dataset):4d} imagenes  {len(train_loader):3d} batches")
print(f"val_loader   : {len(val_loader.dataset):4d} imagenes  {len(val_loader):3d} batches")
print(f"test_loader  : {len(test_loader.dataset):4d} imagenes  {len(test_loader):3d} batches")

In [ ]:
# 6 versiones aumentadas del mismo sample
sample_row  = train_df.iloc[0]
sample_img  = Image.open(faces_root / sample_row["image_path"]).convert("RGB")
player_name = sample_row["label"]

aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
])

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
axes[0].imshow(sample_img.resize((224, 224)))
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(1, 6):
    axes[i].imshow(aug_transform(sample_img))
    axes[i].set_title(f"Aug {i}")
    axes[i].axis("off")

flags_str = f"flip={USE_FLIP}, color_jitter={USE_COLOR_JITTER}"
plt.suptitle(f"Augmentaciones — {player_name}  ({flags_str})", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, lbl in zip(axes.flatten(), images[:8], labels[:8]):
    ax.imshow(denormalize(img))
    ax.set_title(idx_to_label[lbl.item()], fontsize=8)
    ax.axis("off")

plt.suptitle("Batch de entrenamiento — primeras 8 imagenes", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
images, labels = next(iter(train_loader))
total_split = len(train_loader.dataset) + len(val_loader.dataset) + len(test_loader.dataset)

print("=== Verificacion ===")
print(f"Shape del batch   : {images.shape}")
print(f"Dtype             : {images.dtype}")
print(f"Clases            : {len(label_to_idx)}")
print(f"Total en splits   : {total_split}  (esperado: {len(df)})")

assert images.shape == torch.Size([BATCH_SIZE, 3, 224, 224]), "Shape inesperado"
assert total_split == len(df), "El split no suma el total de imagenes"
print("OK — todo consistente")

In [ ]:
# Descargar los archivos para commitear al repo
from google.colab import files
for fname in ["train.csv", "val.csv", "test.csv", "label_to_idx.json"]:
    files.download(f"{DATA_DIR}/{fname}")